## Reading 35-2 - The Power of Ten, Rules 8 through 10, and the Tools That Check Them

By the end of this reading, you will learn:
<ol>
    <li>Rule 8: why macros that look like functions are dangerous</li>
    <li>Rule 9: why flight software restricts pointers, and what that means for portable code</li>
    <li>Rule 10: why "zero warnings" is a rule, not a goal</li>
    <li>How Cobra, a static analyzer developed at NASA's Jet Propulsion Laboratory, checks the Power of Ten automatically, and what it cannot know</li>
</ol>

### Rule 8 - Limit the Preprocessor - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec35/macros.c">macros.c</a>

The preprocessor runs before the compiler and works on <i>text</i>, not on C. That makes macros that look like functions behave very differently from functions:

    #define SQUARE(x) x * x
    #define MAX(a, b) ((a) > (b) ? (a) : (b))

    $ make macros
    SQUARE(offset + 1) = 7
    square(offset + 1) = 16

    MAX(retries++, 2) = 6, retries is now 7
    max(retries++, 2) = 5, retries is now 6

Here is what happened:
<ul>
    <li><code>SQUARE(offset + 1)</code> becomes the text <code>offset + 1 * offset + 1</code>, which is 3 + 3 + 1 = 7. Multiplication happened first.</li>
    <li><code>MAX</code> has all the parentheses it needs, but <code>retries++</code> appears in the substituted text <i>twice</i>, so <code>retries</code> was incremented twice.</li>
</ul>

Neither bug produced a warning. The fix is a <code>static inline</code> function, which the compiler can optimize just as well, and which evaluates each argument exactly once and checks its type:

    static inline int square(int x)
    {
    	return x * x;
    }

Holzmann allows the preprocessor for <code>#include</code>, header guards, and simple constants like <code>#define MAX_COMMANDS 16</code>. His rule specifically forbids token pasting, macros with variable argument lists, and recursive macros. His own <code>c_assert</code> from Reading 35-1 is a function-like macro, and it is acceptable because it stays simple and needs <code>__FILE__</code>, <code>__LINE__</code>, and <code>#e</code>, which only the preprocessor can provide.

### Rule 9 - Restrict Pointers

Holzmann's original rule allows no more than <b>one level of dereferencing</b>, and <b>no function pointers</b>.

You learned pointers to pointers this semester, and they are a legitimate part of C. So why restrict them? Every extra level of indirection makes it harder for a person, <i>or a tool</i>, to determine what memory a line of code can change. A static analyzer that cannot follow a pointer has to assume it could point anywhere, and then it can prove very little.

Function pointers are the extreme case. A call through a function pointer could go to <i>any</i> function with a matching type, so a tool cannot build a complete call graph. Without a call graph, it cannot check Rule 1 (no recursion) or calculate the maximum stack depth.

This has a practical consequence for portable code. Recall from Reading 33-2 that the same BLAS interface can be backed by different implementations for different processors. A common way to choose between them at run time is a table of function pointers, which Rule 9 forbids. The compliant way is to choose at <b>build time</b>, by linking a different library. You did exactly that in Homework 9:

    LIBS := -l:libopenblas.so.0 -lm

Changing that one line in the Makefile changes the implementation, with no function pointers and no change to your code.

### Rule 10 - Zero Warnings, Every Day

Holzmann's final rule: compile with every warning enabled, from the first day of development, and keep the code at <b>zero warnings</b>. Run a static analyzer every day, also with zero warnings.

You have followed this rule since Homework 1. <code>-Werror</code> turns every warning into an error, so a warning cannot be ignored or forgotten. Holzmann's reasoning: if a tool reports something that turns out not to be a bug, rewrite the code so the tool can see that it is correct. Code that confuses a tool usually confuses people too.

Here is where each tool in this course fits:

<table>
    <tr><th>Tool</th><th>Power of Ten rules it helps enforce</th></tr>
    <tr><td><code>-Wall -Wextra -Wconversion -Werror</code></td><td>Rule 10, and many unsafe conversions</td></tr>
    <tr><td><code>-Wshadow</code></td><td>Rule 6</td></tr>
    <tr><td><code>[[nodiscard]]</code> with <code>-Werror</code></td><td>Rule 7, for the functions you mark</td></tr>
    <tr><td><code>-fanalyzer</code></td><td>Rule 3 consequences: leaks, double frees, NULL dereferences</td></tr>
    <tr><td>AddressSanitizer, UBSan, valgrind</td><td>The bugs the rules are meant to prevent, caught while the program runs</td></tr>
    <tr><td>Cobra</td><td>All ten, by pattern</td></tr>
</table>

### Cobra: The Power of Ten as a Tool

Holzmann also built a static analyzer to check his rules. <b>Cobra</b> was developed at JPL, released publicly in 2016, and is now open source. It reads large C programs quickly and checks them against rule sets that ship with the tool, including the Power of Ten, the JPL coding standard, and MISRA C:2012. Space ROS, an open-source robotics framework for space missions, includes Cobra as one of its standard analysis tools.

Here is Cobra's Power of Ten check on the command queue from In-Class Lecture 34, before and after you fixed it:

    $ cobra -f p10 cmd_queue.c                     (the starter)
      1: P10_1, 3 patterns :: Rule 1: restrict to simple control flow constructs
      2: P10_3, 3 patterns :: Rule 3: do not use dynamic memory allocation after initialization
      3: P10_5, 10 patterns :: Rule 5: use minimally two assertions per function on average
      ...
      5: P10_7, 1 patterns :: failure to check validity ptr param

    $ cobra -f p10 cmd_queue.c                     (after Lecture 34)
      1: P10_3, 1 patterns :: Rule 3: do not use dynamic memory allocation after initialization
      2: P10_5, 10 patterns :: Rule 5: use minimally two assertions per function on average
      ...

The Rule 1 violations (the three recursive functions) are gone. Rule 5 is still reported, because the program has no assertions at all, which you now know how to fix. But notice that Rule 3 is <i>still</i> reported, even though the fixed program never calls <code>malloc</code>.

### Running Cobra Yourself

Cobra is installed for our course on <code>student13</code>. After you run <code>scl enable gcc-toolset-15 bash</code>, load it with:

    source /escnfs/courses/fa26-cse-20133.01/public/cobra/cobra_env.sh

You need to run this once each time you log in. Then, from any folder with C code in it:

    p10check cmd_queue.c

<code>p10check</code> runs Cobra's Power of Ten rule set and prints only the summary shown above. Cobra writes the full details, including the line number of every finding, to <code>_P10_.txt</code> in the folder where you ran it. You can also run Cobra directly with <code>cobra -f p10 cmd_queue.c</code>, which prints the same summary along with Cobra's progress messages.

### What a Tool Cannot Know

Why does Cobra still report Rule 3? Its Rule 3 check is a <b>pattern</b>: it reports any function, other than one with <code>init</code> in its name, that calls a function whose name contains <code>alloc</code> or <code>free</code>. Our pool function is named <code>cmd_alloc</code>. Cobra cannot tell that <code>cmd_alloc</code> takes a node from a static array instead of calling <code>malloc</code>.

This is a <b>false positive</b>: a report that is not actually a bug. It is not a flaw in Cobra. Every static analyzer trades precision for speed, and Cobra is designed to check millions of lines in seconds. What matters is what a professional does next:
<ul>
    <li><b>Rename</b> the function (for example, <code>cmd_acquire</code>) so the code no longer looks like what the rule forbids, which follows Holzmann's advice to rewrite code until the tool can see it is correct, or</li>
    <li><b>Document a deviation</b>, as in Reading 34-1: which rule, why the report does not apply, and who reviewed it</li>
</ul>

The one thing you may never do is ignore it. A team that learns to ignore some warnings will eventually ignore one that matters.

Tools also encode a <i>version</i> of a rule. Cobra's rule set allows functions up to 75 lines and two levels of dereferencing, where Holzmann's 2006 paper said 60 lines and one level. Real projects adjust these limits for their mission, and write the adjustment down.

### <font color = "red">Class Introduction Question #4 - Explain why <code>SQUARE(offset + 1)</code> evaluated to 7, and why <code>MAX(retries++, 2)</code> incremented <code>retries</code> twice. What should you write instead?</font>

### <font color = "red">Class Introduction Question #5 - Why does Rule 9 forbid function pointers? How can flight software still choose between different BLAS implementations without them?</font>

### <font color = "red">Class Introduction Question #6 - Cobra reported a Rule 3 violation in code that never calls <code>malloc</code>. Why, and what are the two acceptable ways to respond?</font>

### In class, we will run these tools on a program of your own and bring it to zero warnings under Rules 4 through 10.